# Loan Approval Prediction

This notebook uses the saved loan approval pipeline to predict the outcome for **one new applicant**. The pipeline contains both the preprocessing steps and the trained Decision Tree model, so the applicant record is transformed in exactly the same way as the training data before a prediction is made.

## 1. Import Required Libraries and Locate the Project Files

We use `joblib` to load the saved Scikit-learn pipeline and `pandas` to represent the applicant as a one-row table. `Path` builds a reliable path to the project root, so the notebook can locate the model when it is run from the `notebooks/` directory.

In [ ]:
from pathlib import Path

import joblib
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
MODEL_PATH = PROJECT_ROOT / "models" / "loan_approval_pipeline.pkl"

print(f"Model path: {MODEL_PATH}")

Model path: /home/josephayemlo/Desktop/Software_Development/AI Engineering/Projects/Loan_Approval_Using_Decision_Tree/models/loan_approval_decision_tree.pkl


## 2. Load the Saved Prediction Pipeline

The file `loan_approval_pipeline.pkl` is an end-to-end pipeline, despite its filename. It includes:

* **Preprocessing:** missing-value imputation and categorical encoding.
* **Decision Tree model:** the fitted classifier that produces the loan decision.

> **Why this matters:** Loading the full pipeline prevents the feature-mismatch error that occurs when raw applicant data is sent directly to a model trained on processed features.

In [2]:
saved_pipeline = joblib.load(MODEL_PATH)

print("Saved prediction pipeline loaded successfully.")
print(f"Pipeline steps: {list(saved_pipeline.named_steps)}")

Saved prediction pipeline loaded successfully.
Pipeline steps: ['preprocessing', 'model']


## 3. Define One New Applicant

The new applicant must use the same **33 raw input columns** that were available during training. The values below are an example application; replace them with a real applicant's information when making a new prediction.

Missing values are represented with `None`. The saved preprocessing pipeline will handle supported missing values using the imputation rules learned during training.

In [3]:
new_applicant = {
    "Age": 35,
    "Marital_Status": "Married",
    "Education": "Graduate",
    "Dependents": 2,
    "Residence_Type": "Urban",
    "City_Tier": 2,
    "Employment_Type": "Salaried",
    "Years_at_Current_Job": 5,
    "Total_Work_Experience": 10,
    "Monthly_Income": 180000,
    "Other_Income": 15000.0,
    "Existing_Loans": 1,
    "Existing_Loan_Amount": 500000,
    "Monthly_EMI": 18000,
    "Debt_to_Income": 18.5,
    "Savings": 750000,
    "Investments": 200000.0,
    "Bank_Balance": 350000.0,
    "Credit_Card_Utilization": 22.0,
    "Number_of_Bank_Accounts": 2,
    "Number_of_Credit_Cards": 1,
    "Credit_Score": 760.0,
    "Loan_Defaults": 0,
    "Missed_Payments": 0,
    "Tax_Return_Filed": "Yes",
    "Loan_Purpose": "Home Improvement",
    "Loan_Amount": 1200000,
    "Loan_Tenure": 10,
    "Interest_Rate": 11.5,
    "Collateral": "Yes",
    "Collateral_Value": 2000000.0,
    "Loan_to_Value": 60.0,
    "Total_Debt_Exposure": 1700000,
}

applicant_data = pd.DataFrame([new_applicant])

print(f"Applicant data shape: {applicant_data.shape}")
display(applicant_data)

Applicant data shape: (1, 33)


,Age,Marital_Status,Education,Dependents,Residence_Type,City_Tier,Employment_Type,Years_at_Current_Job,Total_Work_Experience,Monthly_Income,...,Missed_Payments,Tax_Return_Filed,Loan_Purpose,Loan_Amount,Loan_Tenure,Interest_Rate,Collateral,Collateral_Value,Loan_to_Value,Total_Debt_Exposure
0,35,Married,Graduate,2,Urban,2,Salaried,5,10,180000,...,0,Yes,Home Improvement,1200000,10,11.5,Yes,2000000.0,60.0,1700000


### 🔍 Input Schema Verification

Before predicting, we verify that the applicant table has the same columns and column order the pipeline received during training. This check catches missing, unexpected, or incorrectly named fields before they reach the model.

In [4]:
expected_columns = list(saved_pipeline.feature_names_in_)

missing_columns = set(expected_columns) - set(applicant_data.columns)
unexpected_columns = set(applicant_data.columns) - set(expected_columns)

if missing_columns or unexpected_columns:
    raise ValueError(
        f"Input schema mismatch. Missing columns: {sorted(missing_columns)}; "
        f"unexpected columns: {sorted(unexpected_columns)}"
    )

applicant_data = applicant_data[expected_columns]
print("Input schema verified successfully.")
print(f"Number of input features: {applicant_data.shape[1]}")

Input schema verified successfully.
Number of input features: 33


## 4. Generate the Loan Prediction

Calling `predict()` sends the raw applicant data through every preprocessing step and then into the trained Decision Tree. No manual encoding or imputation is required in this notebook.

`predict_proba()` returns the model's probability for each class. These values express the model's relative confidence based on patterns in the training data; they are not a guarantee of a lending outcome.

In [5]:
prediction = saved_pipeline.predict(applicant_data)[0]
probabilities = saved_pipeline.predict_proba(applicant_data)[0]

probability_by_class = pd.Series(
    probabilities,
    index=saved_pipeline.classes_,
    name="Probability",
).sort_values(ascending=False)

print("=" * 60)
print("LOAN APPROVAL PREDICTION")
print("=" * 60)
print(f"\nPredicted loan status: {prediction}")
print("\nPrediction probabilities:")
display(probability_by_class.map(lambda value: f"{value:.2%}").to_frame())

LOAN APPROVAL PREDICTION

Predicted loan status: Approved

Prediction probabilities:


,Probability
Approved,100.00%
Rejected,0.00%


## 5. Prediction Interpretation

The displayed status is the classification produced by the saved Decision Tree for this applicant. The probability table shows how strongly this trained model favors each possible class.

> **Responsible use note:** This notebook is a technical demonstration of model inference. A real loan decision should also use applicable business rules, human review, fairness assessment, and regulatory requirements.